# 03 — Model Development and Selection

Binary classification of cardiac rhythm: Normal (0) vs Abnormal (1).

**Calls:** `src/evaluate.py`
**Input:** `data/processed/physionet_features.csv` (8,187 records × 8 features)
**Output:** `outputs/models/` (saved model + selection report)

**Pre-registered success thresholds** (locked before training):
- Sensitivity ≥ 0.80
- Specificity ≥ 0.75

**Models:** Logistic Regression, Random Forest, XGBoost, SVM
**Split:** 80/10/10 stratified (train / validation / test)

### Evaluation Philosophy — Why Sensitivity Comes First

This is a **screening context**, not a diagnostic one. The model's job is triage: flag anything worth a clinical look, clear everything that is clearly normal.

The asymmetry matters:
- A **false negative** (missed abnormal case) means a person with a real rhythm problem is told they are fine. Direct harm.
- A **false positive** (flagged normal case) means an unnecessary follow-up appointment. Inconvenience, not harm.

**This project therefore prioritises sensitivity over specificity, and both over raw accuracy.** Accuracy is misleading when classes are imbalanced (61.6% Normal) — a model that always predicts Normal scores 61.6% accuracy while catching zero abnormal cases.

**Pre-registered criteria (set before training begins):**

| Criterion | Threshold | Rationale |
|---|---|---|
| Sensitivity | ≥ 80% | Catch at least 8 in 10 true abnormal cases |
| Specificity | ≥ 75% | Correctly clear at least 3 in 4 normal cases |
| AUROC | ≥ 0.85 | Strong overall discriminative ability |

These criteria determine whether a model is deployable for screening — not whether it has the highest accuracy.

## Load Feature Matrix

Reads the feature matrix produced by `02_feature_engineering.ipynb`. Separates features, binary labels, and original Physionet labels (needed for AF-specific secondary analysis in evaluation).

**Input:** `data/processed/physionet_features.csv`

In [1]:
import sys
import os
import subprocess
import pandas as pd
import numpy as np

PROJECT_ROOT = subprocess.check_output(
    ['git', 'rev-parse', '--show-toplevel'], text=True
).strip()
sys.path.insert(0, PROJECT_ROOT)

PROC_DIR = os.path.join(PROJECT_ROOT, 'data', 'processed')

df = pd.read_csv(os.path.join(PROC_DIR, 'physionet_features.csv'))

print(f"Feature matrix loaded: {df.shape[0]:,} records × {df.shape[1]} columns")
print(f"
Class distribution:")
print(f"  Normal   (0): {(df["binary_label"]==0).sum():,} ({(df["binary_label"]==0).mean()*100:.1f}%)")
print(f"  Abnormal (1): {(df["binary_label"]==1).sum():,} ({(df["binary_label"]==1).mean()*100:.1f}%)")
print(f"
Original label breakdown:")
print(df['label'].value_counts().to_string())


Feature matrix loaded: 8,187 records × 11 columns

Class distribution:
  Normal   (0): 5,042 (61.6%)
  Abnormal (1): 3,145 (38.4%)

Original label breakdown:
label
N    5042
O    2391
A     754

Feature matrix X: (8187, 8)
Missing values: 0


## Train / Validation / Test Split

Three-way stratified split: 80% train, 10% validation, 10% test.

- **Train** — model fitting
- **Validation** — threshold optimisation and model comparison
- **Test** — final held-out evaluation (touched once, after model selection)

Stratification preserves the ~62/38 class ratio across all splits. Random state is fixed for reproducibility.

The original Physionet labels (`N`, `A`, `O`) are carried through for AF-specific secondary analysis — they are NOT used during training.

In [2]:
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

# ── First split: 80% train, 20% temp ─────────────────────────
X_train, X_temp, y_train, y_temp, labels_train, labels_temp = train_test_split(
    X, y, labels_original,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

# ── Second split: 50/50 of temp → 10% val, 10% test ──────────
X_val, X_test, y_val, y_test, labels_val, labels_test = train_test_split(
    X_temp, y_temp, labels_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=RANDOM_STATE
)

print("Split Summary")
print("="*55)
print(f"{'Set':<12} {'Total':>7} {'Normal':>8} {'Abnormal':>10} {'Ratio':>8}")
print("-"*55)
for name, yy in [('Train', y_train), ('Validation', y_val), ('Test', y_test)]:
    n0 = (yy == 0).sum()
    n1 = (yy == 1).sum()
    print(f"{name:<12} {len(yy):>7,} {n0:>8,} {n1:>10,} {n1/len(yy)*100:>7.1f}%")
print("-"*55)
print(f"{'Total':<12} {len(y):>7,} {(y==0).sum():>8,} {(y==1).sum():>10,} {(y==1).mean()*100:>7.1f}%")

Split Summary
Set            Total   Normal   Abnormal    Ratio
-------------------------------------------------------
Train          6,549    4,033      2,516    38.4%
Validation       819      504        315    38.5%
Test             819      505        314    38.3%
-------------------------------------------------------
Total          8,187    5,042      3,145    38.4%


## Feature Scaling

StandardScaler fitted on training data only. The same scaler is applied to validation and test sets to prevent data leakage.

Scaling is required for Logistic Regression and SVM (distance-based). Random Forest and XGBoost are tree-based and scale-invariant, but we apply scaling uniformly for consistency across all models.

In [3]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=FEATURE_COLS,
    index=X_train.index
)
X_val_scaled = pd.DataFrame(
    scaler.transform(X_val),
    columns=FEATURE_COLS,
    index=X_val.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=FEATURE_COLS,
    index=X_test.index
)

print("Scaling complete — fitted on training set only.")
print(f"\nTrain mean (should be ~0): {X_train_scaled.mean().mean():.6f}")
print(f"Train std  (should be ~1): {X_train_scaled.std().mean():.6f}")

Scaling complete — fitted on training set only.

Train mean (should be ~0): 0.000000
Train std  (should be ~1): 1.000076


## Model Training

Four classifiers trained on scaled training data with `class_weight='balanced'` where supported. This up-weights the minority class (Abnormal, ~38%) during training to improve sensitivity without manual oversampling.

All models use `probability=True` or equivalent to produce probability scores required for threshold optimisation.

| Model | Why included |
|-------|-------------|
| Logistic Regression | Linear baseline — if this passes, the problem is linearly separable |
| Random Forest | Ensemble of decision trees — handles non-linear boundaries |
| XGBoost | Gradient boosting — typically strongest on tabular data |
| SVM (RBF kernel) | Kernel-based — captures complex decision boundaries |

**Calls:** scikit-learn, xgboost

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

try:
    from xgboost import XGBClassifier
    xgb_available = True
except ImportError:
    xgb_available = False
    print("WARNING: xgboost not installed. Run: pip install xgboost")

print("="*55)
print("MODEL TRAINING")
print("="*55)

models = {}

# ── Logistic Regression ───────────────────────────────────────
print("\n  Training Logistic Regression...", end=" ")
lr = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=RANDOM_STATE
)
lr.fit(X_train_scaled, y_train)
models['Logistic Regression'] = lr
print("done.")

# ── Random Forest ─────────────────────────────────────────────
print("  Training Random Forest...", end=" ")
rf = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf.fit(X_train_scaled, y_train)
models['Random Forest'] = rf
print("done.")

# ── XGBoost ───────────────────────────────────────────────────
if xgb_available:
    print("  Training XGBoost...", end=" ")
    # Compute scale_pos_weight for class imbalance
    n_neg = (y_train == 0).sum()
    n_pos = (y_train == 1).sum()
    xgb = XGBClassifier(
        n_estimators=200,
        scale_pos_weight=n_neg / n_pos,
        eval_metric='logloss',
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
    xgb.fit(X_train_scaled, y_train)
    models['XGBoost'] = xgb
    print("done.")

# ── SVM (RBF kernel) ─────────────────────────────────────────
print("  Training SVM (RBF)...", end=" ")
svm = SVC(
    kernel='rbf',
    class_weight='balanced',
    probability=True,
    random_state=RANDOM_STATE
)
svm.fit(X_train_scaled, y_train)
models['SVM'] = svm
print("done.")

print(f"\n{len(models)} models trained successfully.")

MODEL TRAINING

  Training Logistic Regression... done.
  Training Random Forest... done.
  Training XGBoost... done.
  Training SVM (RBF)... done.

4 models trained successfully.


## Cross-Validation — Performance Stability Check

Five-fold stratified cross-validation on the training set (6,549 records). Each fold independently fits a fresh scaler and applies the same threshold optimisation logic used in final evaluation — this ensures CV estimates reflect the actual pipeline, not a fixed-threshold approximation.

This is a confidence check, not a primary output. If mean sensitivity is stable across folds (low std), the validation set result is not a lucky split artifact.

**Calls:** `src/evaluate.cross_validate_models()`

In [5]:
from src.evaluate import cross_validate_models

# ── Run 5-fold stratified CV with threshold optimisation ──────
# Uses UNSCALED X_train — scaler is fitted fresh per fold inside
cv_results = cross_validate_models(
    models=models,
    X_train=X_train,
    y_train=y_train,
    n_folds=5,
    random_state=RANDOM_STATE
)

print("="*70)
print("5-FOLD STRATIFIED CROSS-VALIDATION (threshold-optimised per fold)")
print("="*70)
print(f"{'Model':<22} {'Sens (mean±std)':>16} "
      f"{'Spec (mean±std)':>16} {'AUROC (mean±std)':>17}")
print("-"*70)

for name, r in cv_results.items():
    print(f"{name:<22} "
          f"{r['sensitivity_mean']:>5.1%} ± {r['sensitivity_std']:.1%}"
          f"   {r['specificity_mean']:>5.1%} ± {r['specificity_std']:.1%}"
          f"   {r['auroc_mean']:>6.4f} ± {r['auroc_std']:.4f}")

print("="*70)
print(f"\nPre-registered floor: Sensitivity ≥ 80.0%")
print(f"Low std = stable performance across folds = trustworthy estimate.")

5-FOLD STRATIFIED CROSS-VALIDATION (threshold-optimised per fold)
Model                   Sens (mean±std)  Spec (mean±std)  AUROC (mean±std)
----------------------------------------------------------------------
Logistic Regression    80.5% ± 0.1%   84.2% ± 2.0%   0.8817 ± 0.0066
Random Forest          80.3% ± 0.2%   86.7% ± 1.3%   0.8955 ± 0.0080
XGBoost                80.4% ± 0.2%   82.5% ± 1.9%   0.8809 ± 0.0054
SVM                    80.3% ± 0.2%   86.9% ± 1.4%   0.8960 ± 0.0062

Pre-registered floor: Sensitivity ≥ 80.0%
Low std = stable performance across folds = trustworthy estimate.


## Train vs Test Accuracy — Overfitting Check

Computes classification accuracy on both the **training set** and **held-out test set** for all four models. This surfaces potential overfitting — a large gap between train and test accuracy indicates the model has memorised training patterns rather than learning generalisable decision boundaries.

Note: Accuracy is reported here for completeness but is **not** the primary evaluation metric. Sensitivity remains the primary metric for this screening context.

**Calls:** nothing from `src/` — inspection only, produces no files

In [6]:
from sklearn.metrics import accuracy_score

print("="*65)
print("TRAIN vs TEST ACCURACY — ALL MODELS")
print("="*65)
print(f"{'Model':<22} {'Train Acc':>10} {'Test Acc':>10} {'Gap':>8}")
print("-"*65)

for name, model in models.items():
    train_acc = accuracy_score(y_train, model.predict(X_train_scaled))
    test_acc  = accuracy_score(y_test, model.predict(X_test_scaled))
    gap = train_acc - test_acc
    print(f"{name:<22} {train_acc:>9.1%} {test_acc:>9.1%} {gap:>+7.1%}")

print("="*65)
print("\nA large positive gap suggests overfitting.")
print("A negative gap suggests the test set was slightly easier.")

TRAIN vs TEST ACCURACY — ALL MODELS
Model                   Train Acc   Test Acc      Gap
-----------------------------------------------------------------
Logistic Regression        83.2%     85.6%   -2.4%
Random Forest             100.0%     86.6%  +13.4%
XGBoost                    99.8%     86.2%  +13.6%
SVM                        85.6%     86.8%   -1.2%

A large positive gap suggests overfitting.
A negative gap suggests the test set was slightly easier.


## Model Evaluation — Validation Set

Evaluates all trained models on the **validation set** using `src/evaluate.py`. For each model:

1. Finds the optimal classification threshold (maximise specificity while meeting sensitivity ≥ 0.80)
2. Computes sensitivity, specificity, AUROC, F1 (Abnormal class)
3. Runs AF-specific secondary analysis (AF is only 23.9% of Abnormal class — the model may catch Other rhythms better than AF)
4. Documents failure mode if the model does not meet the sensitivity floor

**Calls:** `src/evaluate.evaluate_model()`

In [7]:
from src.evaluate import evaluate_model

print("="*55)
print("MODEL EVALUATION — VALIDATION SET")
print("="*55)

reports = []
for name, model in models.items():
    report = evaluate_model(
        name=name,
        model=model,
        X_test=X_val_scaled,
        y_test=y_val,
        label_col=labels_val
    )
    reports.append(report)

MODEL EVALUATION — VALIDATION SET

  LOGISTIC REGRESSION [PASS]
  Optimal threshold : 0.41
  Sensitivity       : 80.3%
  Specificity       : 84.5%
  AUROC             : 0.8810
  F1 (Abnormal)     : 0.7833
  AF sensitivity    : 95.3%
  Confusion matrix  :
    TP=253  FP=78
    FN=62  TN=426

  RANDOM FOREST [PASS]
  Optimal threshold : 0.37
  Sensitivity       : 80.0%
  Specificity       : 86.7%
  AUROC             : 0.9025
  F1 (Abnormal)     : 0.7950
  AF sensitivity    : 98.8%
  Confusion matrix  :
    TP=252  FP=67
    FN=63  TN=437

  XGBOOST [PASS]
  Optimal threshold : 0.34
  Sensitivity       : 80.0%
  Specificity       : 84.7%
  AUROC             : 0.8874
  F1 (Abnormal)     : 0.7826
  AF sensitivity    : 95.3%
  Confusion matrix  :
    TP=252  FP=77
    FN=63  TN=427

  SVM [PASS]
  Optimal threshold : 0.34
  Sensitivity       : 80.0%
  Specificity       : 88.1%
  AUROC             : 0.8981
  F1 (Abnormal)     : 0.8038
  AF sensitivity    : 98.8%
  Confusion matrix  :
    TP=2

## Model Selection — Natural Selection Framework

Applies the five-criterion selection framework to choose the best model for Layer 2:

1. **Survival condition** — eliminate any model below the sensitivity floor (0.80)
2. **Highest specificity** — among survivors, select the model with highest specificity
3. **AUROC tiebreaker** — if specificity is tied, use AUROC to break the tie
4. **Null outcome protocol** — if no model survives, document as a qualified negative finding

**Calls:** `src/evaluate.select_best_model()`

In [8]:
from src.evaluate import select_best_model

selection = select_best_model(reports, models)

if selection['selected_name']:
    print(f"\nProceeding to held-out test evaluation with: {selection['selected_name']}")
else:
    print(f"\nNo model met the pre-registered threshold.")
    print(f"This is a valid negative finding — document and proceed to conclusions.")


  MODEL SELECTION — NATURAL SELECTION FRAMEWORK

  Surviving models:
    SVM: sensitivity=80.0%  specificity=88.1%  AUROC=0.8981
    Random Forest: sensitivity=80.0%  specificity=86.7%  AUROC=0.9025
    XGBoost: sensitivity=80.0%  specificity=84.7%  AUROC=0.8874
    Logistic Regression: sensitivity=80.3%  specificity=84.5%  AUROC=0.8810

  SELECTED: SVM
  Reason: SVM selected. Met sensitivity floor (80.0%) with highest specificity among survivors (88.1%).

Proceeding to held-out test evaluation with: SVM


## Held-Out Test Evaluation — Fixed Threshold

Final evaluation of the selected model (SVM) on the **test set** using the threshold determined on the validation set. The threshold is **not** re-optimised on test data — re-scanning thresholds on the test set would constitute threshold leakage.

The validation-set threshold is applied directly to test set probabilities. Sensitivity and specificity at this fixed threshold are the honest held-out estimates. AUROC is threshold-independent and unaffected.

**Calls:** nothing from `src/` — fixed threshold application only

In [9]:
import numpy as np
from sklearn.metrics import roc_auc_score, f1_score, confusion_matrix

# ── Retrieve the validation-set threshold for SVM ─────────────────
# This threshold was determined on the validation set only.
# It is applied directly to the test set without re-optimisation.
# Re-optimising on the test set would constitute threshold leakage.
svm_val_report = next(
    r for r in reports if r['name'] == 'SVM'
)
fixed_threshold = svm_val_report['threshold']

print("Held-Out Test Set Evaluation — SVM")
print("="*50)
print(f"Fixed threshold (from validation set): {fixed_threshold}")
print(f"Note: threshold not re-optimised on test data.\n")

# ── Apply fixed threshold to test set probabilities ───────────────
svm_model  = models['SVM']
test_proba = svm_model.predict_proba(X_test_scaled)[:, 1]
test_preds = (test_proba >= fixed_threshold).astype(int)

# ── Compute metrics at fixed threshold ────────────────────────────
tn, fp, fn, tp = confusion_matrix(
    y_test, test_preds, labels=[0, 1]
).ravel()

sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
auroc       = roc_auc_score(y_test, test_proba)
f1          = f1_score(y_test, test_preds,
                       pos_label=1, zero_division=0)

# ── AF-specific sensitivity on test set ──────────────────────────
af_mask        = labels_test.reset_index(drop=True) == 'A'
af_proba       = test_proba[af_mask.values]
af_preds       = (af_proba >= fixed_threshold).astype(int)
af_sensitivity = af_preds.mean()

# ── Print final report ────────────────────────────────────────────
print(f"Sensitivity    : {sensitivity:.1%}")
print(f"Specificity    : {specificity:.1%}")
print(f"AUROC          : {auroc:.4f}")
print(f"F1 (Abnormal)  : {f1:.4f}")
print(f"AF sensitivity : {af_sensitivity:.1%}")
print(f"\nConfusion matrix:")
print(f"  TP={tp}  FP={fp}")
print(f"  FN={fn}  TN={tn}")

# ── Meets pre-registered criterion? ──────────────────────────────
meets = sensitivity >= 0.80 and specificity >= 0.75
print(f"\nPre-registered criterion "
      f"(Sens ≥80%, Spec ≥75%): "
      f"{'PASS' if meets else 'FAIL'}")

Held-Out Test Set Evaluation — SVM
Fixed threshold (from validation set): 0.34
Note: threshold not re-optimised on test data.

Sensitivity    : 84.4%
Specificity    : 87.3%
AUROC          : 0.9080
F1 (Abnormal)  : 0.8243
AF sensitivity : 98.8%

Confusion matrix:
  TP=265  FP=64
  FN=49  TN=441

Pre-registered criterion (Sens ≥80%, Spec ≥75%): PASS


## Comparison Summary Table

Side-by-side comparison of all models on the validation set. Highlights which models met the pre-registered sensitivity floor and how they compare on secondary metrics.

In [10]:
print("="*75)
print("MODEL COMPARISON — VALIDATION SET")
print("="*75)
print(f"{'Model':<22} {'Sens':>7} {'Spec':>7} {'AUROC':>7} "
      f"{'F1':>7} {'AF Sens':>8} {'Result':>8}")
print("-"*75)

for r in reports:
    af = f"{r['af_sensitivity']:.1%}" if r['af_sensitivity'] is not None else "N/A"
    status = "PASS" if r['meets_criterion'] else "FAIL"
    print(f"{r['name']:<22} "
          f"{r['sensitivity']:>6.1%} "
          f"{r['specificity']:>6.1%} "
          f"{r['auroc']:>7.4f} "
          f"{r['f1_abnormal']:>7.4f} "
          f"{af:>8} "
          f"{status:>8}")

print("="*75)
print(f"\nPre-registered thresholds: Sensitivity ≥ 80.0%, Specificity ≥ 75.0%")

if selection['selected_name']:
    print(f"Selected model: {selection['selected_name']}")
    print(f"Reason: {selection['reason']}")
else:
    print(f"No model selected — qualified negative finding.")

MODEL COMPARISON — VALIDATION SET
Model                     Sens    Spec   AUROC      F1  AF Sens   Result
---------------------------------------------------------------------------
Logistic Regression     80.3%  84.5%  0.8810  0.7833    95.3%     PASS
Random Forest           80.0%  86.7%  0.9025  0.7950    98.8%     PASS
XGBoost                 80.0%  84.7%  0.8874  0.7826    95.3%     PASS
SVM                     80.0%  88.1%  0.8981  0.8038    98.8%     PASS

Pre-registered thresholds: Sensitivity ≥ 80.0%, Specificity ≥ 75.0%
Selected model: SVM
Reason: SVM selected. Met sensitivity floor (80.0%) with highest specificity among survivors (88.1%).


## Save Model and Report

Saves the selected model (via joblib) and the full evaluation report (as JSON) to `outputs/models/`. If no model was selected, only the null outcome report is saved.

**Output:**
- `outputs/models/selected_model.joblib` — fitted sklearn estimator
- `outputs/models/scaler.joblib` — fitted StandardScaler (needed for Layer 2)
- `outputs/models/evaluation_report.json` — full metrics and selection reason

In [11]:
import json
import joblib

MODEL_DIR = os.path.join(PROJECT_ROOT, 'outputs', 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# ── Custom encoder for numpy types ────────────────────────────
# sklearn/numpy return np.bool_, np.int64, np.float64 which
# json.dump cannot serialise natively.
class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, (np.bool_,)):
            return bool(obj)
        if isinstance(obj, (np.integer,)):
            return int(obj)
        if isinstance(obj, (np.floating,)):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return super().default(obj)

# ── Build test report dict from held-out test results ─────────
# Variables computed in the fixed-threshold held-out test cell.
test_report = {
    'name':             'SVM',
    'threshold':        fixed_threshold,
    'sensitivity':      round(float(sensitivity), 4),
    'specificity':      round(float(specificity), 4),
    'auroc':            round(float(auroc), 4),
    'f1_abnormal':      round(float(f1), 4),
    'meets_criterion':  bool(meets),
    'confusion_matrix': {
        'tp': int(tp), 'tn': int(tn),
        'fp': int(fp), 'fn': int(fn)
    },
    'af_sensitivity':   round(float(af_sensitivity), 4),
    'threshold_source': 'validation_set_fixed'
}

# ── Save evaluation report ────────────────────────────────────
report_data = {
    'validation_reports': reports,
    'selection': {
        'selected_name':  selection['selected_name'],
        'reason':         selection['reason'],
        'eliminated':     selection['eliminated']
    }
}

if selection['selected_name']:
    report_data['test_report'] = test_report

report_path = os.path.join(MODEL_DIR, 'evaluation_report.json')
with open(report_path, 'w') as f:
    json.dump(report_data, f, indent=2, cls=NumpyEncoder)
print(f"Evaluation report saved: {report_path}")

# ── Save model and scaler ─────────────────────────────────────
if selection['selected_name']:
    model_path = os.path.join(MODEL_DIR, 'selected_model.joblib')
    joblib.dump(selection['selected_model'], model_path)
    print(f"Model saved: {model_path}")

    scaler_path = os.path.join(MODEL_DIR, 'scaler.joblib')
    joblib.dump(scaler, scaler_path)
    print(f"Scaler saved: {scaler_path}")
else:
    print("No model saved — null outcome.")

print("\nNotebook complete. Proceed to 04_apple_watch_bridge.ipynb.")

Evaluation report saved: C:\Projects\GA Capstone Project\outputs\models\evaluation_report.json
Model saved: C:\Projects\GA Capstone Project\outputs\models\selected_model.joblib
Scaler saved: C:\Projects\GA Capstone Project\outputs\models\scaler.joblib

Notebook complete. Proceed to 04_apple_watch_bridge.ipynb.


## Save All Four Trained Models

Saves all four Layer 1 models as individual joblib files for cross-model comparison on MIMIC PERform AF data.
The selected model (`selected_model.joblib`) and scaler (`scaler.joblib`) remain unchanged.

**Output:**
- `outputs/models/logistic_regression.joblib`
- `outputs/models/random_forest.joblib`
- `outputs/models/xgboost.joblib`
- `outputs/models/svm.joblib`

In [12]:
# Output: outputs/models/logistic_regression.joblib, random_forest.joblib, xgboost.joblib, svm.joblib

import joblib
import os

MODEL_DIR = os.path.join(PROJECT_ROOT, 'outputs', 'models')

# Map model names to filenames
model_filenames = {
    'Logistic Regression': 'logistic_regression.joblib',
    'Random Forest':       'random_forest.joblib',
    'XGBoost':             'xgboost.joblib',
    'SVM':                 'svm.joblib'
}

print("Saving all four Layer 1 models...")
print("=" * 55)

for name, model in models.items():
    filename = model_filenames[name]
    path = os.path.join(MODEL_DIR, filename)
    joblib.dump(model, path)
    print(f"  {name:<22} → {filename}")

print("=" * 55)
print(f"\n{len(models)} models saved to {MODEL_DIR}")
print("Existing selected_model.joblib and scaler.joblib unchanged.")

Saving all four Layer 1 models...
  Logistic Regression    → logistic_regression.joblib
  Random Forest          → random_forest.joblib
  XGBoost                → xgboost.joblib
  SVM                    → svm.joblib

4 models saved to C:\Projects\GA Capstone Project\outputs\models
Existing selected_model.joblib and scaler.joblib unchanged.


### Layer 1 Complete

The SVM meets all pre-registered criteria on held-out clinical ECG:
- **Sensitivity: 84.4%** — exceeds 80% criterion
- **Specificity: 87.3%** — exceeds 75% criterion
- **AUROC: 0.9080** — exceeds 0.85 criterion
- **AF Sensitivity: 98.8%** — the model almost never misses a true AF case

**But this is only half the question.** The model was trained and tested on clinical ECG recordings. The research question asks whether it generalises to *consumer wearable signals* — a fundamentally different sensing modality (optical PPG vs. electrical ECG).

**Next:** Notebooks 04 and 05 test that generalisation. Notebook 04 applies the SVM to personal Apple Watch data around a confirmed clinical event. Notebook 05 applies it to MIMIC PERform AF — 35 subjects with real PPG recordings and pre-assigned clinical labels.

## Supplementary — Random Forest Feature Importance

Extracts feature importance rankings from the trained Random Forest model. This is a **supplementary analytical finding only** — SVM is the selected Layer 2 model (SVM does not produce native feature importance scores).

RF feature importance shows which of the eight HRV features contributed most to splitting decisions during training. This provides interpretability context for the clinical narrative even though SVM is the deployed model.

**Calls:** nothing from `src/` — inspection and save only
**Output:** `outputs/models/rf_feature_importance.csv`

In [13]:
import pandas as pd
import os

OUTPUTS_DIR = os.path.join(PROJECT_ROOT, 'outputs', 'models')
FEATURE_COLS = [
    'rmssd', 'sdnn', 'mean_rr', 'pnn50',
    'hr_mean', 'hr_std', 'rr_skewness', 'rr_kurtosis'
]

# Extract from trained Random Forest — supplementary analysis
# SVM is the selected Layer 2 model.
# RF feature importance is a secondary analytical finding only.
importance_df = pd.DataFrame({
    'feature':    FEATURE_COLS,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False).reset_index(drop=True)

importance_df['importance_pct'] = (
    importance_df['importance'] * 100
).round(2)

print("Random Forest Feature Importance Rankings")
print("(Supplementary analysis — SVM is Layer 2 model)")
print("="*48)
print(importance_df[['feature','importance_pct']].to_string(
    index=False
))

importance_path = os.path.join(
    OUTPUTS_DIR, 'rf_feature_importance.csv'
)
importance_df.to_csv(importance_path, index=False)
print(f"\nSaved to outputs/models/rf_feature_importance.csv")

Random Forest Feature Importance Rankings
(Supplementary analysis — SVM is Layer 2 model)
    feature  importance_pct
      rmssd           19.58
     hr_std           15.58
    mean_rr           13.32
    hr_mean           12.39
       sdnn           11.87
rr_skewness            9.61
      pnn50            9.31
rr_kurtosis            8.32

Saved to outputs/models/rf_feature_importance.csv


In [14]:
import joblib
import os

OUTPUTS_DIR = os.path.join(PROJECT_ROOT, 'outputs', 'models')

scaler = joblib.load(os.path.join(OUTPUTS_DIR, 'scaler.joblib'))

print("Scaler Verification")
print("="*45)
print(f"Scaler type: {type(scaler).__name__}")
print(f"\nFeature means (fitted on training set only):")
for feat, mean in zip(FEATURE_COLS, scaler.mean_):
    print(f"  {feat:<15}: {mean:.4f}")
print(f"\nIf means look physiologically plausible,")
print(f"scaler was fitted correctly on training data.")

Scaler Verification
Scaler type: StandardScaler

Feature means (fitted on training set only):
  rmssd          : 106.4544
  sdnn           : 88.0448
  mean_rr        : 832.3995
  pnn50          : 0.2563
  hr_mean        : 75.4754
  hr_std         : 10.7064
  rr_skewness    : -0.5065
  rr_kurtosis    : 3.3138

If means look physiologically plausible,
scaler was fitted correctly on training data.
